# 광고 이미지 만들기 4 · FLUX.2 Klein 4B 스케치 검증 v0.2

이 노트북은 **스케치 이미지 1장 + 요청사항**으로 카페 광고용 이미지를 생성합니다. 화면 구현은 포함하지 않습니다.

1. Colab 메뉴에서 `런타임 > 런타임 유형 변경 > L4 GPU`를 선택합니다. 1024×1024 검증에는 약 22.5GB VRAM의 L4를 권장합니다.
2. 개발 6건은 `feature4_flux2_klein_colab_inputs_v0_2.zip`, 테스트 4건은 `feature4_flux2_klein_colab_test4_inputs_v0_2.zip`을 준비합니다.
3. `런타임 > 모두 실행`을 누른 뒤 입력 ZIP 하나를 업로드합니다.
4. 첫 실행에서는 공개 모델 파일을 내려받기 때문에 시간이 걸립니다.
5. 완료되면 개별 결과, 접촉 시트, 프롬프트 기록과 평가표가 ZIP으로 다운로드됩니다.

모델은 공개된 `black-forest-labs/FLUX.2-klein-4B`를 Colab에서 직접 실행합니다. Hugging Face 토큰이나 유료 이미지 생성 API 키는 필요하지 않습니다. 모델이 영어 중심이므로 각 입력 ZIP에는 한국어 요청과 같은 의미의 영어 실행 프롬프트가 함께 들어 있습니다. 개발 6건과 테스트 4건 모두 동일한 방식으로 처리합니다.


In [ ]:
# FLUX.2 Klein 지원을 위해 최신 Diffusers를 설치합니다. Colab 기본 torch는 유지합니다.
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

%pip install -q -U 'git+https://github.com/huggingface/diffusers.git' 'transformers>=4.57,<6' 'accelerate>=1,<2' 'safetensors>=0.6,<1' 'sentencepiece>=0.2,<1' 'hf_xet>=1,<2'


In [ ]:
from __future__ import annotations

import csv
import gc
import json
from math import ceil
from pathlib import Path
import shutil
import time
import zipfile

import torch
from PIL import Image, ImageDraw, ImageFont, ImageOps
from google.colab import files
from IPython.display import Markdown, display

if not torch.cuda.is_available():
    raise RuntimeError('GPU 런타임이 아닙니다. 런타임 유형을 GPU로 변경해주세요.')

GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print('GPU:', GPU_NAME)
print('GPU memory (GB):', round(GPU_MEMORY_GB, 1))
print('Model dtype:', DTYPE)
if GPU_MEMORY_GB < 20:
    print('경고: 이 1024x1024 구성은 L4급 20GB 이상 GPU를 권장합니다. T4 14.6GB에서는 CUDA OOM이 발생할 수 있습니다.')

WORK_DIR = Path('/content/feature4_flux2_klein_v0_2')
INPUT_DIR = WORK_DIR / 'input'
SKETCH_DIR = INPUT_DIR / 'images'
OUTPUT_DIR = WORK_DIR / 'output'
RESULT_DIR = OUTPUT_DIR / 'images'
for directory in (INPUT_DIR, SKETCH_DIR, OUTPUT_DIR, RESULT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

NOTEBOOK_VERSION = 'feature4_flux2_klein_colab_unified_v0.2.1'
MODEL_ID = 'black-forest-labs/FLUX.2-klein-4B'
WIDTH = 1024
HEIGHT = 1024
NUM_INFERENCE_STEPS = 4
GUIDANCE_SCALE = 1.0
print('Notebook version:', NOTEBOOK_VERSION)
print('작업 폴더:', WORK_DIR)


In [ ]:
print('개발 6건 또는 테스트 4건 입력 ZIP 파일 하나를 업로드하세요.')
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise RuntimeError('입력 ZIP 파일 하나만 업로드해주세요.')

zip_path = WORK_DIR / zip_names[0]
zip_path.write_bytes(uploaded[zip_names[0]])
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(INPUT_DIR)

cases_path = INPUT_DIR / 'cases.csv'
font_path = INPUT_DIR / 'NotoSansKR-Variable.ttf'
if not cases_path.is_file() or not SKETCH_DIR.is_dir():
    raise RuntimeError('입력 ZIP에 cases.csv와 images 폴더가 필요합니다.')

with cases_path.open('r', encoding='utf-8-sig', newline='') as handle:
    CASES = list(csv.DictReader(handle))
if not CASES:
    raise RuntimeError('cases.csv에 실행할 사례가 한 건 이상 필요합니다.')

for row in CASES:
    row['seed'] = int(row['seed'])
    row['sketch_path'] = SKETCH_DIR / row['image_filename']
    if not row['sketch_path'].is_file():
        raise FileNotFoundError(row['sketch_path'])

print('입력 확인:', len(CASES), '건')
for row in CASES:
    print(f"- {row['case_id']} · {row['product_name_ko']} · seed={row['seed']}")


In [ ]:
# 스케치는 배치 안내로만 사용하고 선·화살표·네모 상자는 결과에서 제거하도록 공통 규칙을 고정합니다.
INTERNAL_RULES = '''
The uploaded image is a rough layout sketch for a small cafe product advertisement, not a visual style reference and not a finished image.
Before rendering, distinguish intended scene objects from production annotations.
Treat arrows, rectangular outlines, guide boxes, guide lines, labels, and handwritten marks as production annotations by default.
Use annotations only to infer emphasis, arrangement direction, approximate position, relative scale, camera viewpoint, prop placement, and copy-space location.
Do not reproduce the visual shape, color, stroke, outline, or shadow of any annotation in the final image.
Only include a visible arrow, box, panel, frame, sign, or handwritten mark when the primary request explicitly asks for that element in the final advertisement.
When a rectangle indicates copy space, use only its approximate location and extent. Do not render the rectangle itself.
Render copy space as a seamless and natural continuation of the surrounding wall, tabletop, or background. It must not appear as a white patch, differently colored region, translucent overlay, card, panel, sign, frame, border, or box.
Keep copy space visually simple and low-detail. Do not place products, props, highlights, strong shadows, or busy patterns in that area.
Turn the described product into appetizing photorealistic commercial food photography.
Preserve the intended layout but use physically plausible perspective, lighting, materials, shadows, and food texture.
Only add props that are directly required by the request or clearly indicated as intended scene objects.
Do not render letters, words, prices, logos, watermarks, interface elements, or illustration lines.
The final output must look like a real professionally photographed cafe advertisement image.
'''

def build_prompt(request_en: str) -> str:
    return f'''Primary request:
{request_en.strip()}

Composition and rendering rules:
{INTERNAL_RULES.strip()}'''.strip()

def square_rgb(image: Image.Image, size: int = 1024) -> Image.Image:
    image = ImageOps.exif_transpose(image).convert('RGB')
    image.thumbnail((size, size), Image.Resampling.LANCZOS)
    canvas = Image.new('RGB', (size, size), 'white')
    canvas.paste(image, ((size - image.width) // 2, (size - image.height) // 2))
    return canvas

for row in CASES:
    display(Markdown(f"### {row['case_id']} · {row['product_name_ko']}"))
    display(square_rgb(Image.open(row['sketch_path']), 512))
    print('요청사항:', row['request_ko'])
    print('실행 프롬프트:', row['request_en'])


In [ ]:
# 공개 가중치를 Colab에 직접 로드합니다. API 호출이 아니라 로컬 추론입니다.
from diffusers import Flux2KleinPipeline

load_started = time.perf_counter()
pipe = Flux2KleinPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
)
pipe.enable_model_cpu_offload()
if getattr(pipe, 'vae', None) is not None:
    pipe.vae.enable_slicing()
    pipe.vae.enable_tiling()
pipe.set_progress_bar_config(disable=False)
print('모델 로딩 완료 (초):', round(time.perf_counter() - load_started, 1))


In [ ]:
# 6건을 순차 생성합니다. 이미 결과가 있으면 건너뛰므로 런타임 안에서는 중단 후 재실행할 수 있습니다.
records = []
for index, row in enumerate(CASES, start=1):
    output_path = RESULT_DIR / f"{row['case_id']}.png"
    prompt = build_prompt(row['request_en'])
    elapsed = 0.0

    if not output_path.is_file():
        gc.collect()
        torch.cuda.empty_cache()
        print(f"[{index}/{len(CASES)}] 생성 시작: {row['case_id']}")
        sketch = square_rgb(Image.open(row['sketch_path']), WIDTH)
        generator = torch.Generator(device='cuda').manual_seed(row['seed'])
        started = time.perf_counter()
        with torch.inference_mode():
            result = pipe(
                image=sketch,
                prompt=prompt,
                height=HEIGHT,
                width=WIDTH,
                num_inference_steps=NUM_INFERENCE_STEPS,
                guidance_scale=GUIDANCE_SCALE,
                generator=generator,
            ).images[0].convert('RGB')
        elapsed = time.perf_counter() - started
        result.save(output_path, 'PNG', optimize=True)
        del result
        gc.collect()
        torch.cuda.empty_cache()
        print('완료 (초):', round(elapsed, 1))
    else:
        print(f"[{index}/{len(CASES)}] 기존 결과 사용: {row['case_id']}")

    source_preview = square_rgb(Image.open(row['sketch_path']), 512)
    result_preview = Image.open(output_path).convert('RGB').resize((512, 512), Image.Resampling.LANCZOS)
    pair = Image.new('RGB', (1024, 512), 'white')
    pair.paste(source_preview, (0, 0))
    pair.paste(result_preview, (512, 0))
    display(Markdown(f"### {row['case_id']} · 왼쪽 스케치 / 오른쪽 결과"))
    display(pair)

    records.append({
        'case_id': row['case_id'],
        'category': row['category'],
        'product_name_ko': row['product_name_ko'],
        'request_ko': row['request_ko'],
        'request_en': row['request_en'],
        'final_prompt': prompt,
        'seed': row['seed'],
        'elapsed_seconds': round(elapsed, 2),
        'output_file': f"images/{row['case_id']}.png",
    })


In [ ]:
# 비교 접촉 시트, 실행 기록, 육안 평가표와 결과 ZIP을 만듭니다.
cell_size = 300
label_height = 48
pair_columns = 2
sheet_columns = pair_columns * 2
sheet_rows = ceil(len(CASES) / pair_columns)
sheet = Image.new('RGB', (sheet_columns * cell_size, sheet_rows * (cell_size + label_height)), (238, 238, 238))
draw = ImageDraw.Draw(sheet)
font = ImageFont.truetype(str(font_path), 18) if font_path.is_file() else ImageFont.load_default()

for case_index, row in enumerate(CASES):
    pair_col = case_index % pair_columns
    pair_row = case_index // pair_columns
    top = pair_row * (cell_size + label_height)
    left_input = pair_col * 2 * cell_size
    left_result = left_input + cell_size
    source = square_rgb(Image.open(row['sketch_path']), cell_size)
    result = ImageOps.fit(Image.open(RESULT_DIR / f"{row['case_id']}.png").convert('RGB'), (cell_size, cell_size), method=Image.Resampling.LANCZOS)
    sheet.paste(source, (left_input, top))
    sheet.paste(result, (left_result, top))
    draw.text((left_input + 8, top + cell_size + 8), f"{row['case_id']} · 스케치", font=font, fill=(25, 25, 25))
    draw.text((left_result + 8, top + cell_size + 8), f"{row['product_name_ko']} · 결과", font=font, fill=(25, 25, 25))

contact_sheet_path = OUTPUT_DIR / 'source_result_contact_sheet.png'
sheet.save(contact_sheet_path, 'PNG', optimize=True)
display(sheet)

manifest = {
    'experiment_id': f'feature4_flux2_klein_{len(records)}cases_v0.2.0_annotation_policy',
    'model': MODEL_ID,
    'execution': 'Google Colab local inference',
    'hf_token_required': False,
    'count': len(records),
    'generation': {
        'width': WIDTH,
        'height': HEIGHT,
        'steps': NUM_INFERENCE_STEPS,
        'guidance_scale': GUIDANCE_SCALE,
        'dtype': str(DTYPE),
        'gpu': GPU_NAME,
        'gpu_memory_gb': round(GPU_MEMORY_GB, 1),
    },
    'records': records,
}
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

evaluation_rows = [
    'case_id,layout_preserved,guides_removed,product_correct,ad_quality,copy_space_clean,notes'
]
for row in CASES:
    evaluation_rows.append(f"{row['case_id']},,,,,,")
(OUTPUT_DIR / 'manual_evaluation.csv').write_text('\n'.join(evaluation_rows) + '\n', encoding='utf-8-sig')

report = [
    f'# 기능 4 FLUX.2 Klein 스케치 {len(records)}건 검증 v0.2',
    '',
    f'- 모델: `{MODEL_ID}`',
    f'- 생성 크기: `{WIDTH}x{HEIGHT}`',
    f'- 추론 단계: `{NUM_INFERENCE_STEPS}`',
    '- 입력: 스케치 이미지 1장 + 요청사항',
    '- 평가: 구도 유지, 안내선 제거, 상품 정확성, 광고 품질, 문구 공간을 육안으로 확인',
    '',
    '| 사례 | 상품 | 구도 유지 | 안내선 제거 | 상품 정확성 | 광고 품질 | 문구 공간 | 비고 |',
    '|---|---|---|---|---|---|---|---|',
]
for row in CASES:
    report.append(f"| {row['case_id']} | {row['product_name_ko']} |  |  |  |  |  |  |")
(OUTPUT_DIR / 'quality_report.md').write_text('\n'.join(report) + '\n', encoding='utf-8')

archive_base = f'/content/feature4_flux2_klein_{len(records)}cases_v0_2_results'
archive_path = shutil.make_archive(archive_base, 'zip', OUTPUT_DIR)
print('완료:', archive_path)
files.download(archive_path)
